# RL for LLMs
## Learning outcomes

By the end, you should be able to:

- Explain why RLHF uses preference data instead of only supervised labels.
- Build a small preference dataset from sampled model responses.
- Train and sanity-check a scalar reward model.
- Describe how KL regularization stabilizes RL updates for language models.
- Compare reward-model RL, DPO, and verifiable-reward RL on the same task.

## What this notebook covers

| Section | Method | Key idea |
|---------|--------|----------|
| 0 | Setup | Install, load model, and prepare data |
| 1 | Baseline | Zero-shot accuracy before any training |
| 2 | SFT | Supervised fine-tuning on correct solutions |
| 3 | Reward Model (RM) | Train a model to score response quality |
| 4 | GRPO + RM | RL with the learned reward model |
| 5 | DPO | Direct preference optimization (no RL loop) |
| 6 | RLVR | GRPO with a binary verifiable reward |
| 7 | RLVR | GRPO with a richer reward design |
| 8 | Comparison | Side-by-side results across all methods |

All methods share the same base model, the same 500 training problems, and the same 200 held-out evaluation problems.


## How to use this notebook

Run the notebook from top to bottom. Later sections depend on objects created earlier, especially `tokenizer`, `train_ds`, `eval_ds`, `preference_ds`, and `RESULTS`.

### Runtime expectations

- Use a GPU runtime. The notebook is written for memory-conscious training with 4-bit quantization and LoRA.
- The experiment sizes are intentionally small so the whole lesson remains runnable as a course exercise.
- If you rerun only one section, make sure the required model and dataset objects from previous sections still exist in memory.

### Reading convention

Each major topic follows this pattern:

**Theory → Math → Implementation → Observation**

That rhythm is deliberate: first understand the idea, then the objective, then the code, then what to look for in the result.


## Quick terminology reference

| Term | Meaning in this notebook |
|---|---|
| Policy | The language model being optimized (`π_θ`) |
| Reference policy | A frozen model used to keep updates from drifting too far (`π_ref`) |
| Reward model | A scalar scorer trained from chosen/rejected response pairs |
| KL penalty | A constraint that discourages large policy shifts from the reference model |
| Preference pair | A `(prompt, chosen, rejected)` example used by RM and DPO training |
| Verifiable reward | A deterministic checker, such as exact-answer correctness for GSM8K |

Keep this table handy when reading the RL and preference-learning sections.


## Section 0: Setup

Install dependencies, load the model and dataset, and define reusable utilities used by every training method in this lesson.

This setup stage creates the common foundation that keeps all later comparisons fair and directly comparable.


### Code — Install dependencies

The following cell installs the libraries required for SFT, reward modeling, GRPO, and DPO experiments.


In [ ]:
# Install dependencies
!pip install -q trl peft accelerate bitsandbytes datasets transformers einops -U


### Code — Imports and reproducibility

The following cell imports all required packages and sets random seeds for reproducibility.


In [ ]:
import re
import warnings

import numpy as np
import torch
from datasets import Dataset, load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from trl import (
    DPOConfig,
    DPOTrainer,
    GRPOConfig,
    GRPOTrainer,
    RewardConfig,
    RewardTrainer,
    SFTConfig,
    SFTTrainer,
)

warnings.filterwarnings("ignore")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


### 0.1 — Model and tokenizer

We use **Qwen2.5-1.5B-Instruct**, a 1.5B-parameter model that is already instruction-tuned.

Next, we load the tokenizer and model in a way that is consistent across baseline, SFT, RM, DPO, and RLVR experiments.


### Code — Load tokenizer and base model

The following cell initializes tokenizer/model objects and prints a quick tokenizer sanity check.


In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# Qwen does not define a dedicated pad token by default.
# Reusing EOS as PAD is standard for decoder-only models.
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # right padding for causal LM training

print(f"Tokenizer vocab size : {tokenizer.vocab_size:,}")
print(f"Pad token            : {tokenizer.pad_token!r}")

# Load full-precision base model (used for initial inspection)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.config.pad_token_id = tokenizer.pad_token_id

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Model id:", MODEL_ID)
print("Total params:", f"{total_params:,}")
print("Trainable params before PEFT:", f"{trainable_params:,}")
print(f"Weights memory est.  : {total_params * 2 / 1e9:.2f} GB  (weights only)")

# Tokenizer demo
print("\n-- Tokenizer demo --------------------------------")
sample = "Natalia sold clips to 48 friends in April."
tokens = tokenizer(sample, return_tensors="pt")
ids = tokens["input_ids"][0].tolist()
words = tokenizer.convert_ids_to_tokens(ids)
print(f"  Text   : {sample}")
print(f"  Tokens : {words}")
print(f"  IDs    : {ids}")
print(f"  Length : {len(ids)} tokens")


### 0.2 — Dataset: GSM8K

GSM8K contains 8,500 grade-school math word problems with step-by-step solutions.
Each problem has a ground-truth answer embedded after `####`.

We use:
- **500 problems** for training across all methods
- **200 problems** for evaluation (held-out, never used in training)

The same prompts are used for all training methods; only the training signal changes.

Before building splits, we define helper functions for answer extraction and prompt formatting.


### Code — Load and preview GSM8K

The following cell loads GSM8K and shows one sample to verify schema and content.


In [ ]:
# Load GSM8K
gsm8k_raw = load_dataset("gsm8k", "main")
print(f"Train split : {len(gsm8k_raw['train']):,} problems")
print(f"Test split  : {len(gsm8k_raw['test']):,} problems")

# Preview one example
ex = gsm8k_raw["train"][0]
print("\n--- Example problem ---")
print("Question:", ex["question"][:200])
print("Answer:  ", ex["answer"][-60:])


### 0.3 — Helper functions

We now define two utility functions used throughout the notebook:

1. A function that extracts the ground-truth value from GSM8K answers (the text after `####`).
2. A function that formats each question into the chat prompt expected by the model.

These helpers standardize data processing before we construct train and evaluation splits.


### Code — Define extraction and prompt helpers

The following cell implements helper functions that standardize answer parsing and prompt construction.


In [ ]:
def extract_ground_truth(answer_str):
    """Extract the final numeric answer from GSM8K answer string.
    GSM8K embeds the answer as '#### <number>' at the end.
    """
    match = re.search(r'####\s*([\d,\.\-]+)', answer_str)
    if match:
        return match.group(1).replace(',', '').strip()
    return None

def make_prompt(question):
    """Format a GSM8K question as a chat prompt for Qwen2.5-Instruct.
    The system prompt instructs the model to show its reasoning and
    end with #### <answer> so we can extract it consistently.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are a math tutor. Solve the problem step by step. "
                "At the end of your solution, write the final answer as:\n"
                "#### <number>\n"
                "where <number> is just the numeric answer with no units or commas."
            )
        },
        {"role": "user", "content": question}
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


### 0.4 — Build train/eval splits

We sample a fixed train and evaluation subset, then apply the prompt formatting pipeline.

With these standardized splits prepared, we can run a fair baseline before any training.


### Code — Build processed train/eval datasets

The following cell creates fixed train/eval subsets and applies our standardized prompt format.


In [ ]:
# Build train/eval splits
N_TRAIN = 500
N_EVAL  = 200

train_raw = gsm8k_raw["train"].shuffle(seed=42).select(range(N_TRAIN))
eval_raw  = gsm8k_raw["test"].shuffle(seed=42).select(range(N_EVAL))

# Attach ground truth and formatted prompts
def process_split(split):
    records = []
    for item in split:
        gt = extract_ground_truth(item["answer"])
        if gt is None:
            continue
        records.append({
            "question"      : item["question"],
            "answer_full"   : item["answer"],
            "ground_truth"  : gt,
            "prompt"        : make_prompt(item["question"]),
        })
    return Dataset.from_list(records)

train_ds = process_split(train_raw)
eval_ds  = process_split(eval_raw)

print(f"Training problems : {len(train_ds)}")
print(f"Evaluation problems: {len(eval_ds)}")
print("\n--- Formatted prompt (first 400 chars) ---")
print(train_ds[0]["prompt"][:400])


## Section 1 — Baseline: Zero-Shot Evaluation

Before any training, we measure how well Qwen2.5-1.5B-Instruct performs on GSM8K.

This baseline is the anchor for every method comparison in later sections.

**What we measure:** accuracy = fraction of 200 evaluation problems where `extract_answer(response) == ground_truth`.


### Why quantization and LoRA are needed for this course

To keep experiments feasible on available hardware, we combine quantization and LoRA.

Let `N` be the number of parameters and `P` the precision (bytes per parameter). At minimum, these components occupy memory:

- Model weights: `N × P`
- Gradients (full fine-tuning): `N × P`
- Optimizer states (AdamW): `2 × N × P`
- Activations: depends on batch size, sequence length, hidden size, number of layers, and checkpointing settings

With float16, each `1.5B × 2 bytes` component is about `3 GB`. A rough minimum is `4 × 3 = 12 GB` before activation overhead. GRPO+RM additionally needs a reward model copy.

So we use quantization to reduce `P`, and LoRA to reduce the number of trainable parameters.


### 1.1 — Configure quantization and LoRA

We now define the quantization and LoRA configuration used across training stages.

The next code cell turns this memory strategy into reusable configuration objects.


### Code — Configure QLoRA settings

The following cell frees full-precision weights and defines quantization and LoRA configuration used later.


In [ ]:
# QLoRA for efficient fine-tuning
# From the inspection above:
# - The model has 1.5B parameters
# - Full fine-tuning needs ~12+ GB just for weights/gradients/optimizer states

# Free full-precision model first
del model
torch.cuda.empty_cache()
print("Full-precision model freed.")

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Shared LoRA config for training stages
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05


def make_lora_config(task_type=TaskType.CAUSAL_LM):
    return LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type=task_type,
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
    )


### 1.2 — Reusable model loader

We create a helper that loads the same Qwen backbone either as a causal LM or as a sequence-classification model with a single scalar output head (for reward modeling).

This keeps model loading consistent before moving into evaluation and training routines.


### Code — Reusable model-loading utilities

The following cell defines helper functions for loading LM/RM variants and monitoring GPU memory.


In [ ]:
def load_base_model(for_sequence_classification=False, num_labels=1):
    """Load the base Qwen model with 4-bit quantization.

    Args:
        for_sequence_classification: if True, load reward-model head
        num_labels: number of output labels (1 for scalar reward)
    """
    if for_sequence_classification:
        # Same transformer backbone, but with a scalar classification head.
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_ID,
            num_labels=num_labels,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
        )

    model.config.pad_token_id = tokenizer.pad_token_id
    return model


# GPU memory helper
def print_gpu_memory(label=""):
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"[{label}] GPU memory: {used:.1f} / {total:.1f} GB used")


print_gpu_memory("after setup")


### 1.3 — Evaluation helpers

We need two reusable utilities: one to parse the model's final answer and one to compute accuracy on the evaluation split.

These functions provide the shared evaluation protocol for all methods in the rest of the notebook.


### Code — Prediction parsing and accuracy evaluation

The following cell implements the shared evaluation pipeline used after every training stage.


In [ ]:
def extract_predicted_answer(text):
    """Extract the model's predicted final answer from generated text."""
    # Primary extraction: the required #### pattern
    match = re.search(r'####\s*([\d,\.\-]+)', text)
    if match:
        return match.group(1).replace(',', '').strip()

    # Fallback: last standalone number in the response
    numbers = re.findall(r'\b\d+(?:\.\d+)?\b', text)
    return numbers[-1] if numbers else None


@torch.no_grad()
def evaluate_accuracy(model, dataset, max_new_tokens=256, batch_size=4, label=""):
    """Run greedy decoding on a dataset and compute exact-match accuracy."""
    model.eval()

    # Right-padding is useful for training; left-padding is safer for generation.
    tokenizer.padding_side = "left"

    results = []
    correct = 0

    for i in range(0, len(dataset), batch_size):
        batch = dataset[i : i + batch_size]
        prompts = batch["prompt"] if isinstance(batch["prompt"], list) else [batch["prompt"]]
        gts = batch["ground_truth"] if isinstance(batch["ground_truth"], list) else [batch["ground_truth"]]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        ).to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # deterministic evaluation
            temperature=1.0,
            pad_token_id=tokenizer.pad_token_id,
        )

        for j, (out, gt) in enumerate(zip(outputs, gts)):
            # Decode only newly generated tokens.
            input_len = inputs["input_ids"].shape[1]
            generated = tokenizer.decode(out[input_len:], skip_special_tokens=True)
            pred = extract_predicted_answer(generated)
            is_correct = pred == gt

            correct += int(is_correct)
            results.append(
                {
                    "question": batch["question"][j] if isinstance(batch["question"], list) else batch["question"],
                    "predicted": pred,
                    "ground_truth": gt,
                    "correct": is_correct,
                    "generated": generated[:300],
                }
            )

    accuracy = correct / len(dataset)
    print(f"[{label}] Accuracy: {correct}/{len(dataset)} = {accuracy:.1%}")

    # Restore right-padding for training cells.
    tokenizer.padding_side = "right"
    return accuracy, results


### 1.4 — Run baseline evaluation

We first evaluate the quantized base model with no additional training.

This baseline anchors every improvement reported in later sections.


### Code — Baseline evaluation

The following cell loads the quantized base model and computes baseline held-out accuracy.


In [ ]:
# Load base model for baseline evaluation
print("Loading base model for baseline evaluation...")
base_model = load_base_model()
print_gpu_memory("after loading base model")

# Run baseline evaluation
baseline_acc, baseline_results = evaluate_accuracy(
    base_model, eval_ds, label="Baseline (no training)"
)

# Show a few examples
print("\n--- Sample predictions ---")
for r in baseline_results[:3]:
    status = "✓" if r["correct"] else "✗"
    print(f"{status} GT={r['ground_truth']} | Pred={r['predicted']}")
    print(f"  Q: {r['question'][:80]}...")
    print()

RESULTS = {"baseline": baseline_acc}
print(f"\nBaseline accuracy stored: {baseline_acc:.1%}")


### Checkpoint — Baseline

At this point, you have a reproducible zero-shot score. Every later method should be interpreted relative to this number, not in isolation.

Before moving on, make sure you understand:

- Why exact-match answer extraction is used for GSM8K.
- Why quantization changes memory cost but not the training objective.
- Why the same evaluation set must be reused for every method.


## Section 2 — Supervised Fine-Tuning (SFT)

Now we fine-tune with SFT. We load the base model with quantization, add LoRA adapters, and train only the LoRA parameters.

We train on 500 correct `(question, full_solution)` pairs from GSM8K using a chat template that requires the final answer format `#### <number>`.

**Loss:** standard cross-entropy on assistant tokens.

With this supervised baseline in place, we can later compare preference-based RL methods.


### Code — Prepare SFT training examples

The following cell converts GSM8K items into chat-style supervised examples.


In [ ]:
def make_sft_example(item):
    """Format a training example for SFT.
    The full solution (with step-by-step reasoning) is the target.
    We format as a complete conversation so the model learns the chat template.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are a math tutor. Solve the problem step by step. "
                "At the end of your solution, write the final answer as:\n"
                "#### <number>"
            )
        },
        {"role": "user",      "content": item["question"]},
        {"role": "assistant", "content": item["answer_full"]},
    ]
    return {"messages": messages}

sft_train = train_ds.map(make_sft_example)
sft_train = sft_train.select_columns(["messages"])
print("SFT training example (first 500 chars):")
print(sft_train[0]["messages"][:500])


### 2.1 — Load LoRA model and start SFT

We load a fresh quantized base model, attach LoRA adapters, and prepare for SFT optimization.

The next cell configures training hyperparameters and launches supervised fine-tuning.


### Code — Initialize SFT model with LoRA

The following cell loads a fresh base model, attaches LoRA adapters, and reports trainable parameters.


In [ ]:
# Free base model memory before SFT training
del base_model
torch.cuda.empty_cache()
print_gpu_memory("after freeing base model")

# Load fresh model for SFT
sft_model = load_base_model()
sft_model = get_peft_model(sft_model, make_lora_config())
sft_model.print_trainable_parameters()
print_gpu_memory("after loading SFT model")


### Code — Configure and run SFT

The following cell defines SFT hyperparameters and launches supervised fine-tuning.


In [ ]:
# With 500 examples and batch size 16 (effective), one epoch = ~31 gradient
#update steps. 4 epochs = ~124 steps total.

# Print training metrics (loss, learning rate, gradient norm) to the console
# every 20 gradient update steps.

sft_config = SFTConfig(
    num_train_epochs=4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,   # effective batch = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    gradient_checkpointing=True,
    logging_steps=20,
    save_strategy="epoch",
    max_length=512,
    report_to="none",
)

sft_trainer = SFTTrainer(
    model=sft_model,
    args=sft_config,
    train_dataset=sft_train,
    processing_class=tokenizer,
)

print("Starting SFT training...")
sft_trainer.train()


### Code — Evaluate SFT

The following cell evaluates the SFT model and stores the result for later comparison.


In [ ]:
# Evaluate SFT
sft_acc, _ = evaluate_accuracy(sft_model, eval_ds, label="After SFT")
RESULTS["sft"] = sft_acc
print(f"\nImprovement over baseline: {sft_acc - RESULTS['baseline']:+.1%}")


### 2.2 — Interpreting SFT performance

SFT improves behavior with only adapter updates, but training duration and hardware limits still constrain final accuracy.

Next, we inspect a sample output format before moving to preference-based learning.


### Code — Inspect one SFT generation

The following cell prints a qualitative example output to inspect reasoning format and answer extraction.


In [ ]:
# Look at a few raw SFT outputs to see what format the model is producing
sample = eval_ds[0]
inputs = tokenizer(sample["prompt"], return_tensors="pt").to(sft_model.device)

with torch.no_grad():
    outputs = sft_model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

input_len = inputs["input_ids"].shape[1]
generated = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
print("Question:")
print(sample["question"])
print("Generated output:")
print(generated)
print("\nExtracted answer:", extract_predicted_answer(generated))
print("Ground truth:    ", sample["ground_truth"])


### Checkpoint — SFT

SFT teaches the model to imitate correct worked solutions, but it does not directly optimize preferences or verifiable reward.

Before moving on, make sure you can distinguish:

- The supervised target (`answer_full`) from the generated answer.
- Cross-entropy training from reward-based optimization.
- Format learning from mathematical correctness.


### 2.3 — Transition to RL-based alignment

The sample output shows useful reasoning behavior, but SFT alone can plateau.

Next, we move to reinforcement-learning-style optimization using preference signals.


## Section 3 — Training the Reward Model

In this section (and the next), we train GRPO + RM. The first step is to train a reward model (RM).

The RM takes `(prompt, response)` as input and outputs a scalar reward representing how preferable the response is.

### How we construct preference pairs from GSM8K

For each training problem:
1. Generate 8 candidate solutions using the SFT model (with sampling)
2. For each candidate, check whether `extract_answer(candidate) == ground_truth`
3. Label correct solutions as chosen and incorrect ones as rejected
4. Form `(chosen, rejected)` pairs

### Architecture

The RM uses the same Qwen2.5-1.5B backbone with a linear head that outputs one scalar reward score.


### 3.1 — Pairwise reward-model objective (math)

This equation defines the pairwise preference loss used to train the reward model.

$$
\mathcal{L}_{\mathrm{RM}} = -\log \sigma\left(r_{\text{chosen}} - r_{\text{rejected}}\right)
$$

Where:
- `\sigma` is the sigmoid function
- `r_{chosen}` is the scalar reward assigned to the preferred response
- `r_{rejected}` is the scalar reward assigned to the non-preferred response

This objective pushes chosen responses to score higher than rejected responses.


### Code — Build preference pairs

The following cell samples candidate responses and constructs chosen/rejected preference pairs.


In [ ]:
@torch.no_grad()
def generate_candidates(model, prompt, n=8, max_new_tokens=256, temperature=0.8):
    """Generate `n` candidate solutions for one prompt using sampling."""
    model.eval()
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=0.9,
        num_return_sequences=n,
        pad_token_id=tokenizer.pad_token_id,
    )

    input_len = inputs["input_ids"].shape[1]
    return [tokenizer.decode(out[input_len:], skip_special_tokens=True) for out in outputs]


def build_preference_pairs(model, dataset, n_candidates=8, max_problems=400):
    """Build (prompt, chosen, rejected) pairs from sampled candidates."""
    pairs = []
    skipped = 0

    for i, item in enumerate(dataset):
        if i >= max_problems:
            break

        if i % 50 == 0:
            print(f"  Generating candidates: {i}/{min(max_problems, len(dataset))}")

        candidates = generate_candidates(model, item["prompt"], n=n_candidates)
        gt = item["ground_truth"]

        chosen_list = [c for c in candidates if extract_predicted_answer(c) == gt]
        rejected_list = [c for c in candidates if extract_predicted_answer(c) != gt]

        if not chosen_list or not rejected_list:
            skipped += 1
            continue

        # Keep one valid pair per prompt in this educational example.
        pairs.append(
            {
                "prompt": item["prompt"],
                "chosen": chosen_list[0],
                "rejected": rejected_list[0],
            }
        )

    print(f"\nPairs constructed: {len(pairs)} (skipped {skipped} problems with no valid pair)")
    return Dataset.from_list(pairs)


### 3.2 — Generate the preference dataset

We now create `(prompt, chosen, rejected)` pairs by sampling multiple responses and comparing them against ground truth.

This dataset is the direct training signal for the reward model.


### Code — Generate preference dataset

The following cell runs pair construction and previews one resulting preference example.


In [ ]:
print("Generating candidates with SFT model to build preference pairs...")

preference_ds = build_preference_pairs(
    sft_model, train_ds, n_candidates=8, max_problems=400
)

# Show one pair
ex = preference_ds[0]
print("\n--- Preference pair example ---")
print("PROMPT (last 100 chars):", ex["prompt"][-100:])
print("\nCHOSEN (first 200 chars) :")
print(ex["chosen"][:200])
print("\nREJECTED (first 200 chars):")
print(ex["rejected"][:200])


### 3.3 — Prepare reward-model training

Because the SFT model does not solve every sampled prompt correctly, only a subset of problems yields valid chosen/rejected pairs.

With this filtered preference dataset ready, we now load and train the reward model.


### Code — Initialize reward model

The following cell loads the sequence-classification head and attaches LoRA for RM training.


In [ ]:
# Free SFT model to reclaim memory
del sft_model
torch.cuda.empty_cache()
print_gpu_memory("after freeing SFT model")

# Load RM model (classification head)
print("Loading reward model architecture...")
rm_model = load_base_model(for_sequence_classification=True, num_labels=1)
rm_model = get_peft_model(rm_model, make_lora_config(task_type=TaskType.SEQ_CLS))
rm_model.print_trainable_parameters()
print_gpu_memory("after loading RM model")


### Code — Format RM training examples

The following cell converts preference pairs into chosen/rejected text sequences expected by RewardTrainer.


In [ ]:
def format_rm_example(item):
    """Format a preference pair for the reward model.
    The RM sees 'prompt + response' as a single sequence and outputs a scalar.
    We create two sequences per pair: one for chosen, one for rejected.
    """

    return {
        "chosen":   item["prompt"] + item["chosen"]   + tokenizer.eos_token,
        "rejected": item["prompt"] + item["rejected"] + tokenizer.eos_token,
    }

rm_train_ds = preference_ds.map(format_rm_example,
                                remove_columns=preference_ds.column_names,
                                load_from_cache_file=False)
print(f"RM training examples: {len(rm_train_ds)}")


### Code — Train reward model

The following cell configures RewardTrainer and optimizes pairwise ranking loss.


In [ ]:
rm_config = RewardConfig(
    num_train_epochs=6,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    gradient_checkpointing=True,
    logging_steps=20,
    save_strategy="epoch",
    max_length=512,
    report_to="none",
)

rm_trainer = RewardTrainer(
    model=rm_model,
    args=rm_config,
    train_dataset=rm_train_ds,
    #processing_class=tokenizer,
)

print("Training reward model...")
print("Bradley-Terry loss: -log σ(r_chosen - r_rejected)")
print("Metric to watch: rewards/accuracies — should climb above 0.5 and toward 0.8+")
rm_trainer.train()


### 3.4 — Interpreting reward-model loss

A decreasing pairwise loss indicates the model is learning to rank chosen responses above rejected ones.

Next, we verify this directly by comparing scores on an example pair.


### Code — Sanity-check reward scores

The following cell verifies that chosen responses score higher than rejected responses on a sample pair.


In [ ]:
# Verify that the reward model gives sensible scores
@torch.no_grad()
def score_response(rm, prompt, response):
    """Get scalar reward score for a (prompt, response) pair."""
    text = prompt + response + tokenizer.eos_token
    enc  = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(rm.device)
    out  = rm(**enc)
    return out.logits.squeeze().item()

# Test: correct solution should score higher than wrong solution
ex = preference_ds[0]
score_chosen   = score_response(rm_model, ex["prompt"], ex["chosen"])
score_rejected = score_response(rm_model, ex["prompt"], ex["rejected"])

print(f"Score for chosen   response: {score_chosen:.3f}")
print(f"Score for rejected response: {score_rejected:.3f}")
print(f"Gap: {score_chosen - score_rejected:.3f} (positive = reward model working correctly)")


### Checkpoint — Reward model

The reward model is useful only if it ranks chosen responses above rejected responses often enough to guide policy learning.

Before moving on, make sure you understand:

- Why preference pairs avoid asking for absolute scalar labels.
- Why only score differences matter in the pairwise loss.
- Why a reward model can be imperfect and still useful.


## Section 4 — GRPO+RM: RL with the learned reward model

Now that the reward model is trained, we run GRPO + RM.

### Models used in this stage

| Model | Role | Trainable? |
|-------|------|-----------|
| Policy (`\pi_\theta`) | The LLM being optimized | ✓ LoRA adapters |
| Reference (`\pi_{ref}`) | Frozen baseline model | ✗ frozen |
| Reward model | Scores responses | ✗ frozen |

In practice, we could use the SFT model as `\pi_{ref}`; here we use the base model for simplicity.

### Training loop (per step)

1. Sample a batch of prompts
2. Policy generates responses (rollouts)
3. Reward model scores each response
4. Compute normalized advantages
5. Update policy with clipped objective


### 4.1 — GRPO objective and advantage normalization (math)

This objective regularizes reward optimization with a KL penalty to the reference policy.

$$
\text{maximize: } \mathbb{E}[r(s,a)] - \beta \times \mathrm{KL}(\pi_\theta \parallel \pi_{ref})
$$

Where:
- `r(s,a)` is the reward assigned to the sampled response
- `\beta` scales the KL penalty
- `\mathrm{KL}(\pi_\theta \parallel \pi_{ref})` keeps the policy close to the reference

GRPO estimates advantage from a group of responses for the same prompt.

$$
\text{advantage}_i = \frac{\text{reward}_i - \text{mean(group\_rewards)}}{\text{std(group\_rewards)}}
$$

This normalization gives positive advantage to above-group responses and negative advantage to below-group responses.


### Code — Prepare GRPO+RM dataset and reward function

The following cell builds GRPO inputs and defines a reward callback that queries the trained RM.


In [ ]:
# Dataset for GRPO + RM
def format_grpo_rm(item):
    return {
        "prompt"      : item["prompt"],   # used by GRPOTrainer for generation
        "prompt_text" : item["prompt"],   # passed as kwarg to reward function
    }

grpo_rm_ds = train_ds.map(format_grpo_rm, remove_columns=[
    c for c in train_ds.column_names if c not in ["prompt", "prompt_text"]
])

# Reward function using the extra column
def grpo_rm_reward_fn(completions, prompt_text, **kwargs):
    """
    prompt_text arrives as a list (one per completion in the group batch).
    Same structure as ground_truth in the verifiable reward function.
    """
    rewards = []
    for prompt, completion in zip(prompt_text, completions):
        score = score_response(rm_model, prompt, completion)
        rewards.append(float(score))
    return rewards


### 4.2 — Metrics to monitor during GRPO + RM

During training, the primary metric is mean reward. A rising reward trend indicates that policy updates are improving responses under the learned reward model.

We also track reward standard deviation and KL divergence to monitor learning signal quality and policy drift.


### Code — GRPO metrics logger

The following cell defines a callback to track loss, reward statistics, and KL divergence during training.


In [ ]:
from transformers import TrainerCallback


class GRPOMetricsLogger(TrainerCallback):
    def __init__(self):
        self.history = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return

        # Log keys can vary by trainer/version; use safe fallbacks.
        row = {
            "step": state.global_step,
            "loss": logs.get("loss", float("nan")),
            "reward_mean": logs.get("reward", logs.get("train/reward", float("nan"))),
            "reward_std": logs.get("reward_std", logs.get("train/reward_std", float("nan"))),
            "kl": logs.get("kl", logs.get("train/kl", float("nan"))),
        }
        self.history.append(row)

        print(
            f"Step {row['step']:4d} | "
            f"loss={row['loss']:.6f} | "
            f"reward_mean={row['reward_mean']:.4f} | "
            f"reward_std={row['reward_std']:.4f} | "
            f"kl={row['kl']:.6f}"
        )


grpo_logger = GRPOMetricsLogger()


### 4.3 — Train GRPO + RM

We now train the policy with GRPO using the frozen reward model from Section 3.

After training, we will inspect reward, loss, and KL trends.


### Code — Train GRPO with learned RM

The following cell configures GRPO and runs policy optimization under the learned reward model.


In [ ]:

grpo_rm_config = GRPOConfig(
    num_generations=8,
    beta=0.04,
    learning_rate=1e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    max_steps=400,
    max_completion_length=256,
    temperature=0.8,
    logging_steps=10,
    save_strategy="no",
    bf16=True,
    gradient_checkpointing=False,
    report_to="none",
)

grpo_rm_model = load_base_model()

grpo_rm_logger = GRPOMetricsLogger()

grpo_rm_trainer = GRPOTrainer(
    model=grpo_rm_model,
    args=grpo_rm_config,
    reward_funcs=grpo_rm_reward_fn,
    train_dataset=grpo_rm_ds,
    eval_dataset=grpo_rm_ds.select(range(30)),
    processing_class=tokenizer,
    peft_config=make_lora_config(),
    callbacks=[grpo_rm_logger],
)

print("GRPO + Reward Model training")
print("Reward source: neural network (rm_model) — same reward model from Section 3")
print("This demonstrates classical RLHF reward signal with modern GRPO algorithm")
print()
grpo_rm_trainer.train()
print("Training complete.")


### 4.4 — Interpreting early stopping

Training was stopped early when both mean reward and KL began to plateau, indicating limited additional progress under the current constraint.

Next, we visualize the training curves to diagnose reward growth and policy drift.


### Code — Visualize GRPO+RM training dynamics

The following cell plots reward, reward variance, loss, and KL trends for GRPO+RM.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# Extract clean history (remove NaN entries)
history = [
    h for h in grpo_rm_logger.history
    if not any(np.isnan(v) if isinstance(v, float) else False
               for v in h.values())
    and not np.isnan(h["reward_mean"])
]

steps       = [h["step"]        for h in history]
losses      = [h["loss"]        for h in history]
rewards     = [h["reward_mean"] for h in history]
reward_stds = [h["reward_std"]  for h in history]
kls         = [h["kl"]         for h in history]

def rolling_avg(values, window=5):
    result = []
    for i in range(len(values)):
        start = max(0, i - window + 1)
        result.append(np.mean(values[start:i+1]))
    return result

fig = plt.figure(figsize=(14, 10))
fig.suptitle("GRPO + Reward Model Training — Qwen2.5-1.5B on GSM8K",
             fontsize=14, fontweight="bold", y=0.98)
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

# Plot 1: reward mean
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(steps, rewards, color="#8B5CF6", alpha=0.3, linewidth=1, label="Raw")
ax1.plot(steps, rolling_avg(rewards, 5), color="#8B5CF6", linewidth=2.5,
         label="Rolling avg (5)")
ax1.axhline(rewards[0], color="gray", linestyle="--", linewidth=1.5,
            label=f"Start ({rewards[0]:.2f})")
ax1.axhline(np.mean(rewards[-10:]), color="#8B5CF6", linestyle=":",
            linewidth=1.5, label=f"End avg ({np.mean(rewards[-10:]):.2f})")
ax1.set_title("Reward Mean\n(RM score — higher = better)", fontweight="bold")
ax1.set_xlabel("Training step")
ax1.set_ylabel("Mean reward (RM score)")
ax1.legend(fontsize=9)
ax1.grid(alpha=0.25)
ax1.spines[["top","right"]].set_visible(False)
# Annotate the improvement
improvement = np.mean(rewards[-10:]) - rewards[0]
ax1.annotate(f"Improvement: {improvement:+.2f}",
             xy=(steps[-1], np.mean(rewards[-10:])),
             xytext=(steps[len(steps)//2], rewards[0] + 0.2),
             fontsize=9, color="#8B5CF6",
             arrowprops=dict(arrowstyle="->", color="#8B5CF6"))

# Plot 2: reward std
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(steps, reward_stds, color="#F59E0B", alpha=0.3, linewidth=1)
ax2.plot(steps, rolling_avg(reward_stds, 5), color="#F59E0B", linewidth=2.5,
         label="Rolling avg (5)")
ax2.axhline(0, color="red", linestyle="--", linewidth=1, label="Zero = no signal")
ax2.set_title("Reward Std (within-group variance)\nHigh = rich learning signal",
              fontweight="bold")
ax2.set_xlabel("Training step")
ax2.set_ylabel("Reward std")
ax2.legend(fontsize=9)
ax2.grid(alpha=0.25)
ax2.spines[["top","right"]].set_visible(False)
ax2.text(0.02, 0.95,
         f"Mean std: {np.mean(reward_stds):.2f}\n"
         f"(vs GRPO+verifiable: ~0.08)",
         transform=ax2.transAxes, fontsize=9, va="top",
         bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                   alpha=0.8, edgecolor="#F59E0B"))

# Plot 3: loss
ax3 = fig.add_subplot(gs[1, 0])
clipped = [max(-0.05, min(0.05, l)) for l in losses]
ax3.plot(steps, clipped, color="#3B82F6", alpha=0.3, linewidth=1, label="Raw (clipped)")
ax3.plot(steps, rolling_avg(clipped, 5), color="#3B82F6", linewidth=2.5,
         label="Rolling avg (5)")
ax3.axhline(0, color="gray", linestyle="--", linewidth=1, alpha=0.5)
ax3.set_title("Policy Gradient Loss", fontweight="bold")
ax3.set_xlabel("Training step")
ax3.set_ylabel("Loss")
ax3.legend(fontsize=9)
ax3.grid(alpha=0.25)
ax3.spines[["top","right"]].set_visible(False)

# Plot 4: KL divergence
ax4 = fig.add_subplot(gs[1, 1])
ax4.plot(steps, kls, color="#10B981", linewidth=2.5, label="KL divergence")
ax4.fill_between(steps, 0, kls, color="#10B981", alpha=0.15)
ax4.set_title("KL Divergence from Reference\nShould rise slowly and plateau",
              fontweight="bold")
ax4.set_xlabel("Training step")
ax4.set_ylabel("KL")
ax4.legend(fontsize=9)
ax4.grid(alpha=0.25)
ax4.spines[["top","right"]].set_visible(False)

plt.show()


### What to observe

- Mean reward should trend upward, indicating improvement under the reward model.
- Reward standard deviation indicates how much within-group learning signal remains.
- KL should rise gradually and then stabilize as policy updates are constrained.
- Loss oscillations are expected in policy-gradient training and should be interpreted alongside reward/KL.


### 4.5 — Transition from plots to evaluation

The plotted trends suggest the policy is learning under the RM signal.

Next, we measure end-task accuracy on the held-out evaluation set.


### Code — Evaluate GRPO+RM

The following cell evaluates GRPO+RM performance on held-out GSM8K problems.


In [ ]:
# Evaluate GRPO+RM
grpo_rm_acc, _ = evaluate_accuracy(
    grpo_rm_model, eval_ds, label="After GRPO+RM"
)

# Save to RESULTS
RESULTS["grpo_rm"] = grpo_rm_acc


### Checkpoint — GRPO + RM

GRPO+RM demonstrates the classic RLHF shape: sample responses, score them with a frozen reward model, and update the policy under a KL constraint.

Before moving on, make sure you can explain:

- Why KL limits reward hacking and policy drift.
- Why reward standard deviation matters for group-relative advantage.
- Why held-out accuracy is the final check, not training reward alone.


### 4.6 — Why this result matters

Even a small, imperfect reward model can provide a useful optimization signal that improves downstream accuracy.

Next, we compare this approach against DPO, which removes the reward-model training step.


## Section 5 — DPO: Direct Preference Optimization

DPO removes explicit reward-model training and directly optimizes the policy on preference pairs.

We reuse the same `(prompt, chosen, rejected)` dataset created in Section 3.

This gives a simpler optimization pipeline with no rollout-time reward model.


### 5.1 — DPO loss (math)

This equation defines the direct preference objective used in DPO.

$$
L_{\mathrm{DPO}} = -\log \sigma\left(
\beta \times \log\frac{\pi_\theta(y_w\mid x)}{\pi_{ref}(y_w\mid x)}
- \beta \times \log\frac{\pi_\theta(y_l\mid x)}{\pi_{ref}(y_l\mid x)}
\right)
$$

Where:
- `y_w` is the chosen response (winner)
- `y_l` is the rejected response (loser)
- `\beta` controls how strongly the policy is constrained relative to `\pi_{ref}`

The objective increases relative likelihood of chosen responses and decreases relative likelihood of rejected responses.


### Code — Initialize DPO model

The following cell frees prior models, loads a fresh policy, and attaches LoRA for DPO.


In [ ]:
# Free GRPO+RM model
del grpo_rm_model
torch.cuda.empty_cache()
print_gpu_memory("after freeing PPO model")

# Load fresh model for DPO
print("Loading model for DPO...")
dpo_model = load_base_model()
dpo_model = get_peft_model(dpo_model, make_lora_config())
dpo_model.print_trainable_parameters()
print_gpu_memory("after loading DPO model")


### Code — Configure and run DPO

The following cell defines DPO hyperparameters and trains directly on preference pairs.


In [ ]:
dpo_config = DPOConfig(
    # KL-constraint strength in the DPO objective.
    beta=0.1,  # moderate constraint; typical range: 0.05-0.3

    # Training
    num_train_epochs=2,
    per_device_train_batch_size=2,  # DPO uses chosen + rejected sequences
    gradient_accumulation_steps=8,  # effective batch size = 16
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    bf16=True,
    gradient_checkpointing=True,

    # Sequence length
    max_length=512,

    # Logging/checkpointing
    logging_steps=10,
    save_strategy="no",
    report_to="none",
)

dpo_trainer = DPOTrainer(
    model=dpo_model,
    ref_model=None,  # PEFT: frozen base weights used as reference
    args=dpo_config,
    train_dataset=preference_ds,
    processing_class=tokenizer,
)

print("Starting DPO training...")
dpo_trainer.train()
print("DPO training complete.")
print_gpu_memory("after DPO")


### 5.2 — Interpreting early DPO progress

This run was intentionally short for compute reasons, so the result reflects partial optimization rather than full convergence.

Next, we evaluate DPO on the same held-out evaluation split.


### Code — Evaluate DPO

The following cell evaluates DPO and compares gains against baseline and SFT.


In [ ]:
# Evaluate DPO
dpo_acc, _ = evaluate_accuracy(dpo_model, eval_ds, label="After DPO")
RESULTS["dpo"] = dpo_acc
print(f"Improvement over baseline : {dpo_acc - RESULTS['baseline']:+.1%}")
print(f"Improvement over SFT      : {dpo_acc - RESULTS['sft']:+.1%}")


### Code — Inspect aggregate results

The following cell prints the current `RESULTS` dictionary for all completed stages.


In [ ]:
RESULTS


### Checkpoint — DPO

DPO uses the same preference data as the reward model, but skips explicit reward-model training and rollout-time reward scoring.

Before moving on, make sure you can explain:

- What the chosen and rejected responses represent.
- How the reference-policy ratio appears in the DPO objective.
- Why DPO is often simpler to run than RLHF with a learned reward model.


## Section 6 — GRPO with verifiable rewards (RLVR)

In GRPO + RM, we trained a separate neural reward model. RLVR uses a hard-coded, verifiable reward function instead, so no reward model is needed.

Here the reward is deterministic: correct answer = 1, incorrect answer = 0.

This gives us a direct baseline for rule-based reinforcement learning.


### Code — Initialize GRPO model for RLVR

The following cell loads a fresh policy model for verifiable-reward training.


In [ ]:
# Free DPO model
del dpo_model
torch.cuda.empty_cache()
print_gpu_memory("after freeing DPO model")

# Load fresh model for GRPO
print("Loading model for GRPO...")
grpo_model = load_base_model()
print_gpu_memory("after loading GRPO model")


### 6.1 — Define the binary verifiable reward

We now implement a deterministic reward function based only on correctness.

After defining this reward, we train GRPO and inspect learning behavior.


### Code — Define binary RLVR reward

The following cell defines the rule-based correctness reward and formats the GRPO training dataset.


In [ ]:
# Verifiable reward function for GRPO
def grpo_reward_fn(completions, ground_truth, **kwargs):
    """
    Called by GRPOTrainer after sampling G completions per prompt.

    Args:
        completions: list of G generated strings for this prompt
        ground_truth: the correct answer string

    Returns:
        list of G scalar rewards (1.0 = correct, 0.0 = wrong)

    This is RLVR: a deterministic rule-based checker.
    Cannot be reward-hacked — either the number matches or it doesn't.
    """
    rewards = []
    for completion, gt in zip(completions, ground_truth):
        predicted = extract_predicted_answer(completion)
        reward = 1.0 if (predicted is not None and predicted == gt) else 0.0
        rewards.append(reward)

    return rewards

# Format dataset for GRPO
# GRPOTrainer expects 'prompt' column and any extra columns passed to reward_fn
def format_grpo(item):
    return {
        "prompt"        : item["prompt"],
        "ground_truth"  : item["ground_truth"],
    }

grpo_train_ds = train_ds.map(format_grpo, remove_columns=[
    c for c in train_ds.column_names if c not in ["prompt","ground_truth"]
])


### Code — Train GRPO with binary verifiable reward

The following cell configures and trains GRPO under deterministic correctness feedback.


In [ ]:
grpo_config = GRPOConfig(
    # Core GRPO hyperparameters
    num_generations=8,           # G: number of responses sampled per prompt
                                # More = better advantage estimate, but slower
    # KL constraint
    beta=0.04,                # keeps policy from drifting too far from reference
    # Training
    learning_rate=1e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    max_steps=300,
    # Generation settings
    max_completion_length=256,
    temperature=1.4,             # sampling temperature for rollouts
    # Logging
    logging_steps=10,
    save_strategy="no",
    bf16=True,
    report_to="tensorboard",
)

grpo_trainer = GRPOTrainer(
    model=grpo_model,
    args=grpo_config,
    reward_funcs=grpo_reward_fn,
    train_dataset=grpo_train_ds,
    processing_class=tokenizer,
    peft_config=make_lora_config(),
    callbacks=[grpo_logger],
)

print("Starting GRPO training...")
grpo_trainer.train()
print("GRPO training complete.")
print_gpu_memory("after GRPO")


### 6.2 — Plot RLVR training metrics

We visualize reward statistics, loss, and KL divergence to check whether training is progressing under the binary verifier.

Then we evaluate the trained model on held-out GSM8K samples.


### Code — Visualize binary-RLVR training dynamics

The following cell plots reward, reward variance, loss, and KL for the binary-reward run.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# Extract history from logger
history = [h for h in grpo_logger.history if not any(
    np.isnan(v) for v in [h['reward_mean'], h['reward_std'], h['kl']]
)]

steps       = [h["step"]        for h in history]
losses      = [h["loss"]        for h in history]
rewards     = [h["reward_mean"] for h in history]
reward_stds = [h["reward_std"]  for h in history]
kls         = [h["kl"]         for h in history]

# Rolling average helper
def rolling_avg(values, window=3):
    result = []
    for i in range(len(values)):
        start = max(0, i - window + 1)
        result.append(np.mean(values[start:i+1]))
    return result

fig = plt.figure(figsize=(14, 10))
fig.suptitle("GRPO Training Metrics — Qwen2.5-1.5B on GSM8K",
             fontsize=14, fontweight="bold", y=0.98)
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

# Plot 1: reward mean
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(steps, rewards, color="#10B981", alpha=0.3, linewidth=1, label="Raw")
ax1.plot(steps, rolling_avg(rewards), color="#10B981", linewidth=2.5, label="Rolling avg (3)")
ax1.set_title("Reward Mean", fontweight="bold")
ax1.set_xlabel("Training step")
ax1.set_ylabel("Mean reward (per rollout)")
ax1.legend(fontsize=9)
ax1.grid(alpha=0.25)
ax1.spines[["top","right"]].set_visible(False)

# Plot 2: reward std
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(steps, reward_stds, color="#F59E0B", alpha=0.3, linewidth=1, label="Raw")
ax2.plot(steps, rolling_avg(reward_stds), color="#F59E0B", linewidth=2.5, label="Rolling avg (3)")
ax2.axhline(0, color="red", linestyle="--", linewidth=1.5, label="Zero = no learning")
ax2.set_title("Reward Std  (within-group variance)", fontweight="bold")
ax2.set_xlabel("Training step")
ax2.set_ylabel("Reward std")
ax2.legend(fontsize=9)
ax2.grid(alpha=0.25)
ax2.spines[["top","right"]].set_visible(False)

# Add annotation explaining what zero means
ax2.text(0.98, 0.08,
         "std=0 → all rollouts same reward\n→ zero gradient signal",
         transform=ax2.transAxes, fontsize=8, color="gray",
         ha="right", va="bottom",
         bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                   alpha=0.8, edgecolor="gray"))

# Plot 3: loss
ax3 = fig.add_subplot(gs[1, 0])
# Clip extreme values for visibility
clipped_loss = [max(-0.05, min(0.1, l)) for l in losses]
ax3.plot(steps, clipped_loss, color="#3B82F6", alpha=0.3, linewidth=1, label="Raw (clipped)")
ax3.plot(steps, rolling_avg(clipped_loss), color="#3B82F6", linewidth=2.5, label="Rolling avg (3)")
ax3.axhline(0, color="gray", linestyle="--", linewidth=1, alpha=0.5)
ax3.set_title("Policy Gradient Loss", fontweight="bold")
ax3.set_xlabel("Training step")
ax3.set_ylabel("Loss")
ax3.legend(fontsize=9)
ax3.grid(alpha=0.25)
ax3.spines[["top","right"]].set_visible(False)
ax3.text(0.98, 0.92,
         "Negative loss = normal in PG\n(advantage × log_prob sum)",
         transform=ax3.transAxes, fontsize=8, color="gray",
         ha="right", va="top",
         bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                   alpha=0.8, edgecolor="gray"))

# Plot 4: KL divergence
ax4 = fig.add_subplot(gs[1, 1])
ax4.plot(steps, kls, color="#8B5CF6", linewidth=2.5, label="KL divergence")
ax4.fill_between(steps, 0, kls, color="#8B5CF6", alpha=0.15)
ax4.set_title("KL Divergence from Reference", fontweight="bold")
ax4.set_xlabel("Training step")
ax4.set_ylabel("KL")
ax4.legend(fontsize=9)
ax4.grid(alpha=0.25)
ax4.spines[["top","right"]].set_visible(False)
ax4.text(0.98, 0.08,
         "KL rising = model updating\nKL flat = no weight change",
         transform=ax4.transAxes, fontsize=8, color="gray",
         ha="right", va="bottom",
         bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                   alpha=0.8, edgecolor="gray"))

plt.show()


### What to observe

- Binary rewards can make mean reward noisy; this is expected.
- Reward standard deviation near zero means little advantage signal in sampled groups.
- KL growth indicates the policy is moving away from the reference model.
- Use final held-out accuracy to judge real downstream benefit.


### 6.3 — Interpreting binary-reward trends

With binary rewards, mean reward can look noisy or flat, but KL growth still indicates policy updates relative to the reference model.

Next, we evaluate GRPO-RLVR directly on the evaluation set.


### Code — Evaluate GRPO-RLVR

The following cell evaluates the binary-reward GRPO model.


In [ ]:
# Evaluate GRPO
grpo_acc, _ = evaluate_accuracy(grpo_model, eval_ds, label="After GRPO")
RESULTS["grpo"] = grpo_acc
print(f"Improvement over baseline : {grpo_acc - RESULTS['baseline']:+.1%}")
print(f"Improvement over SFT      : {grpo_acc - RESULTS['sft']:+.1%}")


### Checkpoint — Binary RLVR

Binary RLVR replaces subjective preference scoring with a deterministic correctness checker.

Before moving on, make sure you understand:

- Why verifiable rewards are especially natural for math and code tasks.
- Why binary rewards can be sparse or noisy during training.
- Why KL can still show learning even when reward curves are choppy.


### 6.4 — Transition to richer reward design

A binary correctness reward already improves performance, but it provides limited learning signal granularity.

Next, we add a richer reward with style/format incentives.


## Section 7 — RLVR: Improving the reward

To improve RLVR signal quality, we add a small format/reasoning bonus on top of correctness.

Correctness remains the dominant objective; format rewards only provide gentle shaping.


### 7.1 — Richer reward design (concept)

Conceptually, we use:

`total_reward = correctness_reward + format_bonus`

- `correctness_reward` is the primary term
- `format_bonus` rewards reasoning-style structure (multi-step format, arithmetic traces, and self-check cues)

Next, we implement this reward function directly in code.


### Code — Initialize model for richer RLVR

The following cell loads a fresh policy model for richer reward shaping.


In [ ]:
# Free GRPO model
del grpo_model
torch.cuda.empty_cache()
print_gpu_memory("after freeing GRPO model")

# Load fresh model for RLVR
print("Loading model for RLVR...")
rlvr_model = load_base_model()
print_gpu_memory("after loading RLVR model")


### 7.2 — Implement the richer reward function

We now encode the richer RLVR reward used in this section.

Then we train and evaluate to compare against the binary-reward variant.


### Code — Define richer RLVR reward

The following cell implements correctness plus format-based reward components.


In [ ]:
def rlvr_reward_fn(completions, ground_truth, **kwargs):
    """Richer RLVR reward = correctness + small format bonuses."""
    rewards = []

    for completion, gt in zip(completions, ground_truth):
        # Component 1: correctness (dominant signal)
        predicted = extract_predicted_answer(completion)
        correctness = 1.0 if (predicted is not None and predicted == gt) else 0.0

        # Component 2: format/reasoning bonuses (small shaping terms)
        lines = [l.strip() for l in completion.split("\n") if l.strip()]
        has_multi_step = len(lines) >= 3
        has_arithmetic = bool(re.search(r'[=÷×+\-]\s*\d+', completion))
        has_selfcheck = any(
            phrase in completion.lower()
            for phrase in ["wait", "let me check", "let me verify", "actually", "so the answer is"]
        )

        format_bonus = 0.0
        if has_multi_step:
            format_bonus += 0.05
        if has_arithmetic:
            format_bonus += 0.03
        if has_selfcheck:
            format_bonus += 0.02

        total_reward = correctness + format_bonus
        rewards.append(total_reward)

    return rewards


# Reuse same dataset format as Section 6 (prompt + ground_truth)
rlvr_train_ds = grpo_train_ds


### Code — Train GRPO with richer reward

The following cell runs RLVR training with the richer reward function.


In [ ]:

rlvr_config = GRPOConfig(       # RLVR uses GRPO algorithm, different reward function
    num_generations=8,
    beta=0.04,
    learning_rate=1e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    max_steps=200,
    max_completion_length=300,          # slightly longer to allow more reasoning
    temperature=0.9,
    logging_steps=10,
    save_strategy="no",
    bf16=True,
    report_to="none",
)

rlvr_trainer = GRPOTrainer(
    model=rlvr_model,
    args=rlvr_config,
    reward_funcs=rlvr_reward_fn,    # richer reward vs GRPO section
    train_dataset=rlvr_train_ds,
    processing_class=tokenizer,
    peft_config=make_lora_config()
)

print("Starting RLVR training...")
rlvr_trainer.train()
print("RLVR training complete.")
print_gpu_memory("after RLVR")


### Code — Evaluate richer-RLVR model

The following cell evaluates the richer-reward model and stores the result.


In [ ]:
# Evaluate RLVR
rlvr_acc, rlvr_results = evaluate_accuracy(rlvr_model, eval_ds, label="After RLVR")
RESULTS["rlvr"] = rlvr_acc
print(f"Improvement over baseline : {rlvr_acc - RESULTS['baseline']:+.1%}")
print(f"Improvement over SFT      : {rlvr_acc - RESULTS['sft']:+.1%}")



### 7.3 — Interpreting richer-reward results

The richer reward provides additional shaping signal beyond pure correctness.

Next, we summarize all methods side by side.


### Code — Probe emergent self-check behavior

The following cell measures how often generated responses include self-checking language.


In [ ]:
# Check if self-checking behavior emerged
print("\n--- Checking for emergent reasoning behaviors ---")
selfcheck_count = sum(
    1 for r in rlvr_results
    if any(p in r["generated"].lower() for p in ["wait","let me check","actually","let me verify"])
)
print(f"Responses with self-checking language: {selfcheck_count}/{len(rlvr_results)} ({selfcheck_count/len(rlvr_results):.0%})")


### Checkpoint — Richer RLVR

The richer reward adds small shaping terms while keeping correctness as the dominant signal.

Before moving on, make sure you understand:

- Why auxiliary format rewards should stay smaller than correctness rewards.
- How reward shaping can encourage useful reasoning structure.
- Why shaping terms must be chosen carefully to avoid optimizing style over correctness.


## Section 8 — Comparison: All Methods Side by Side

Now we aggregate all results and compare how each training strategy changed held-out accuracy.

This final section closes the lesson by connecting the full pipeline from baseline to RLVR.


### Code — Print side-by-side result table

The following cell summarizes final accuracies and deltas versus baseline.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Results summary
print("=" * 55)
print(f"{'Method':<20} {'Accuracy':>10} {'vs Baseline':>12}")
print("=" * 55)
for method, acc in RESULTS.items():
    delta = acc - RESULTS["baseline"] if method != "baseline" else 0.0
    marker = "★" if acc == max(RESULTS.values()) else " "
    print(f"{marker} {method:<19} {acc:>9.1%} {delta:>+11.1%}")
print("=" * 55)
print(f"  Best method: {max(RESULTS, key=RESULTS.get).upper()}")


### Code — Plot final method comparison

The following cell visualizes all methods on one bar chart for quick comparison.


In [ ]:
# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))

methods = list(RESULTS.keys())
accs = [RESULTS[m] for m in methods]
colors = ["#94A3B8", "#3B82F6", "#8B5CF6", "#F59E0B", "#10B981", "#0D9488"]

bars = ax.bar(methods, accs, color=colors, width=0.6, edgecolor="white", linewidth=1.5)

# Baseline reference line
ax.axhline(
    RESULTS["baseline"],
    color="#EF4444",
    linestyle="--",
    linewidth=1.5,
    alpha=0.7,
    label=f"Baseline ({RESULTS['baseline']:.1%})",
)

# Value labels on bars
for bar, acc in zip(bars, accs):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.003,
        f"{acc:.1%}",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
    )

ax.set_ylim(0, min(1.0, max(accs) * 1.15))
ax.set_ylabel("Accuracy on GSM8K eval set", fontsize=12)
ax.set_title("GSM8K Accuracy Comparison Across Training Methods", fontsize=13, fontweight="bold")
ax.set_xticklabels([m.upper() for m in methods], fontsize=11)
ax.legend(fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


### What to observe

- Compare absolute held-out accuracy across methods, not just training-time rewards.
- Improvements over baseline/SFT indicate whether preference or verifiable rewards added value.
- The best bar identifies the most effective method under this compute/data setup.


## Final interpretation guide

| Method | What it tests | Main tradeoff |
|---|---|---|
| Baseline | What the instruction-tuned model can already do | No task-specific learning |
| SFT | Imitation from correct demonstrations | Requires target solutions and may plateau |
| Reward model + GRPO | Learning from sampled preference signal | Needs a separate reward model and careful KL control |
| DPO | Direct optimization from preference pairs | Simpler pipeline, no online reward model |
| Binary RLVR | Rule-based correctness optimization | Strong when answers are verifiable, but reward can be sparse |
| Richer RLVR | Correctness plus small shaping rewards | More signal, but shaping must not dominate correctness |

The important comparison is not only which method wins here, but why each method changes the learning signal available to the model.
